# Exact transitions and aggregated signed walkers

The CPU cells check literal LSB matrices. GPU cells run only with explicit
opt-in and an MC-enabled extension. Each GPU smoke uses at most two qubits,
two steps, one trajectory and a fixed seed. No measurement circuits execute.


In [ ]:
import os
import numpy as np
import qarpx as qx
from qarp.algorithms import AggregatedMonteCarlo, ResidentEvolutionConfig
from qarp.blocks import SimpleBlock, TrotterAnsatzBlock
from qarp.engines import CudaqEngine
from qarp.operators import ComputationalBasisSector, QubitOperator
from sympy import Symbol

x = np.array([[0, 1], [1, 0]], dtype=complex)
y = np.array([[0, -1j], [1j, 0]], dtype=complex)
h = (np.kron(x, x) + np.kron(y, y)) / 2
sector_matrix = h[np.ix_([1, 2], [1, 2])]
np.testing.assert_array_equal(sector_matrix, x)
# Each certain spawn flips sign. Population [8, -8] becomes [16, -16].
populations = np.array([8, 0], dtype=int)
for _ in range(2):
    populations = populations - np.array([[0, 1], [1, 0]]) @ populations
assert populations.tolist() == [16, -16]
assert populations[1] / populations[0] == -1
print("Independent CPU matrix and walker oracle passed.")


In [ ]:
gpu_enabled = os.environ.get("QARP_RUN_CUDAQ_GPU_TESTS") == "1" and qx.montecarlo_available()
if not gpu_enabled:
    print("GPU complete-row and request smokes unverified: opt in with an MC-enabled build.")
else:
    u = SimpleBlock(1)
    complete = AggregatedMonteCarlo(
        QubitOperator("X0"), 0.0, 2.0, 1.0, 0, u,
        engine=CudaqEngine(), initial_walker_count=8, shift_damping=0.0,
        num_trajectories=1, column_batch_size=2, seed=7,
    ).build()
    np.testing.assert_allclose(complete.run(), [-1.0], rtol=0, atol=1e-12)
    assert complete.diagnostics["measurement_circuits_executed"] == 0
    assert complete.diagnostics["estimator"]["trajectory_jackknife_stderr"] is None
    print("GPU complete-row oracle passed.")


In [ ]:
if gpu_enabled:
    theta = Symbol("theta")
    identity = TrotterAnsatzBlock(2, [QubitOperator(())], [theta]).set_symbols({theta: 0.0})
    config = ResidentEvolutionConfig(
        ComputationalBasisSector(((0, 1),), (1,)),
        transition_batch_size=4, cache_max_bytes=4096,
    )
    request = AggregatedMonteCarlo(
        (QubitOperator("X0 X1") + QubitOperator("Y0 Y1")) / 2,
        0.0, 2.0, 1.0, 1, identity,
        engine=CudaqEngine(), initial_walker_count=8, shift_damping=0.0,
        num_trajectories=1, resident_evolution=config, seed=7,
    ).build()
    np.testing.assert_allclose(request.run(), [-1.0], rtol=0, atol=1e-12)
    assert request.diagnostics["actual_shots_executed"] == 0
    print("GPU request-transport oracle passed.")


Per-trajectory block jackknife is opt-in through an explicit block size on
ratio-of-sums estimation. Across independent trajectories, the final uncertainty
always uses leave-one-trajectory replicas of the pooled signed ratio. These
single-trajectory smokes therefore have no across-trajectory error bar.
